## Tests vérification entre fichiers et versions

In [1]:
# Pour trace :
# Attention soucis umap avec derniere version numpy
# puis crash kernel
# nouvel env
# puis voulais pas afficher les sorties vizu de bert_topic
# Mais exécuter 2 fois les cellules à marché
# dark magic. J'abandonne.

In [2]:
# Test vérif si taille contexte modèle bien pris en compte par ollama

import os
from ollama import Client

OLLAMA_HOST = os.environ.get("OLLAMA_HOST")
client = Client(host=OLLAMA_HOST)

MODEL = "qwen3-embedding:8b"

# vérif config
print(type(client))
print(client._client.base_url)  # ou selon la version, client._base_url

# Texte suffisamment long (~10k-20k tokens suivant le tokenizer)
text = ("hello " * 15000).strip()

for num_ctx in (4096, 8192, 32768):
    print(f"\n=== num_ctx={num_ctx} ===")

    try:
        client.embed(
            model=MODEL,
            input=text,
            truncate=False,
            options={"num_ctx": num_ctx},
        )
        print("SUCCESS")
    except Exception as e:
        print(f"ERROR: {e}")


<class 'ollama._client.Client'>
https://llm-tools-alpha.huma-num.fr/LM_uyum6AzwsP_fHdxdLg5Gw/

=== num_ctx=4096 ===
ERROR: the input length exceeds the context length (status code: 400)

=== num_ctx=8192 ===
ERROR: the input length exceeds the context length (status code: 400)

=== num_ctx=32768 ===
SUCCESS


In [3]:
import json
import numpy as np
from datasets import load_from_disk
from numpy.linalg import norm


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return np.dot(a, b) / (norm(a) * norm(b))


def compare_embeddings(emb_a: np.ndarray, emb_b: np.ndarray, tolerance: float = 1e-5):
    assert emb_a.shape == emb_b.shape, (
        f"Shapes différentes : {emb_a.shape} vs {emb_b.shape}"
    )

    exact = np.array_equal(emb_a, emb_b)
    close = np.allclose(emb_a, emb_b, atol=tolerance)
    cos_sims = np.array(
        [cosine_similarity(emb_a[i], emb_b[i]) for i in range(len(emb_a))]
    )

    print(f"Identiques (exact)        : {exact}")
    print(f"Identiques (tolérance)    : {close}")
    print(f"Similarité cosinus moyenne: {cos_sims.mean():.6f}")
    print(f"Similarité cosinus min    : {cos_sims.min():.6f}")

    return exact, close, cos_sims


def load_first_n_embeddings_jsonl(path, n: int) -> np.ndarray:
    embeddings = []
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            embeddings.append(json.loads(line))
    return np.array(embeddings)

In [4]:
# si veut vérifier sur un fichier embeddings.jsonl brut,
# sinon on peut comparer les 2 datasets (avec et sans truncation)

# N = 1000
# EMBEDDINGS_JSONL = "../models/embeddings_checkpoint/embeddings.jsonl"

# # Depuis le JSONL (calcul brut)
# emb_jsonl = load_first_n_embeddings_jsonl(EMBEDDINGS_JSONL, N)

# # Depuis le Dataset final (après assemblage)
# dataset = load_from_disk(FINAL_DATASET_DIR)
# subset = dataset.select(range(N))
# emb_dataset = np.array(subset["embedding"])

# exact, close, cos_sims = compare_embeddings(emb_jsonl, emb_dataset)

In [ ]:
# définition des chemins
truncate = load_from_disk(
    "../models/embeddings/qwen3-8b_embeddings_2026-07-21/dataset_with_embeddings"
)
truncate = np.array(truncate["embedding"])

no_truncate = load_from_disk("../models/embeddings/dataset_with_embeddings/")
no_truncate = np.array(no_truncate["embedding"])

# test
exact, close, cos_sims = compare_embeddings(truncate, no_truncate)

Identiques (exact)        : True
Identiques (tolérance)    : True
Similarité cosinus moyenne: 1.000000
Similarité cosinus min    : 1.000000
